# 04 Model Training (Kaggle)

Run this notebook in Kaggle after cloning the repo into `/kaggle/working/Trading-with-the-Momentum-Transformer`. It checks the project layout, downloads the raw CSV if needed, updates runtime paths, runs the full training pipeline, and archives outputs in `/kaggle/working`.

In [ ]:
import os
from pathlib import Path

print('Working dir:', Path.cwd())
!nvidia-smi || true

In [ ]:
REPO_DIR = '/kaggle/working/Trading-with-the-Momentum-Transformer'
RAW_CSV_URLS = [
    'https://raw.githubusercontent.com/votranhuonggiang/Trading-with-the-Momentum-Transformer/exp/full-sell-tax-cost/data/raw/vn30f1m.csv',
    'https://raw.githubusercontent.com/votranhuonggiang/Trading-with-the-Momentum-Transformer/exp/full-sell-tax-cost/vn30f1m_momentum_transformer/data/raw/vn30f1m.csv',
]

cwd = Path.cwd()
candidate_dirs = [
    cwd,
    Path(REPO_DIR),
    cwd / 'vn30f1m_momentum_transformer',
    Path(REPO_DIR) / 'vn30f1m_momentum_transformer',
]

PROJECT_DIR = None
for candidate in candidate_dirs:
    if (candidate / 'configs').is_dir() and (candidate / 'src').is_dir():
        PROJECT_DIR = str(candidate)
        break

if PROJECT_DIR is None:
    checked = [str(p) for p in candidate_dirs]
    raise RuntimeError(f'Cannot find project root (expected configs/ and src/). Checked: {checked}')

print('PROJECT_DIR =', PROJECT_DIR)
os.chdir(PROJECT_DIR)
!pwd
!ls
!git branch --show-current
!git log --oneline -n 3

In [ ]:
from pathlib import Path
import urllib.request

csv_path = Path(PROJECT_DIR) / 'data' / 'raw' / 'vn30f1m.csv'
csv_path.parent.mkdir(parents=True, exist_ok=True)

if not csv_path.exists():
    downloaded = False
    for url in RAW_CSV_URLS:
        try:
            urllib.request.urlretrieve(url, csv_path)
            if csv_path.exists():
                downloaded = True
                break
        except Exception:
            pass
    if not downloaded:
        raise RuntimeError('Could not download vn30f1m.csv from GitHub raw URLs.')

print('Ready input:', csv_path)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import yaml

cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
cfg.setdefault('paths', {})
cfg.setdefault('trading', {})
cfg.setdefault('models', {})
cfg.setdefault('walk_forward', {})
cfg.setdefault('training', {})

# Kaggle-safe paths while keeping the current scenario settings from GitHub.
cfg['paths']['project_root'] = PROJECT_DIR
cfg['paths']['raw_data_csv'] = 'data/raw/vn30f1m.csv'
cfg['models']['main_models'] = ['lstm_dmn', 'decoder_transformer', 'decoder_tft']
cfg['walk_forward'].pop('max_windows', None)
cfg['training'].pop('max_epochs_per_window', None)
cfg['training'].pop('max_train_rows_per_window', None)
cfg['training'].pop('max_valid_rows_per_window', None)

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('Updated config for Kaggle run:', cfg_path)

In [ ]:
!python src/data_preprocessing.py --config configs/default.yaml
!python src/feature_engineering.py --config configs/default.yaml
!python src/walk_forward.py --config configs/default.yaml
!python src/evaluate.py --config configs/default.yaml

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pred_path = Path('outputs/predictions/all_predictions.csv')
ts = pd.read_csv(pred_path, parse_dates=['timestamp']).sort_values(['model', 'timestamp'])
ts['cum_net_return'] = ts.groupby('model')['net_return'].cumsum()

fig_dir = Path('outputs/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / 'model_cumulative_returns.png'

plt.figure(figsize=(14, 7))
for m, g in ts.groupby('model'):
    plt.plot(g['timestamp'], g['cum_net_return'], label=m, linewidth=1.6)
plt.title('Cumulative Net Return by Model')
plt.xlabel('Timestamp')
plt.ylabel('Cumulative Net Return')
plt.grid(alpha=0.25)
plt.legend(loc='best', ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()
print('Saved figure:', fig_path)


In [ ]:
!ls -lah outputs/metrics
!ls -lah outputs/tables
!ls -lah outputs/figures

In [ ]:
import shutil
from pathlib import Path

zip_base = '/kaggle/working/vn30f1m_outputs'
zip_file = shutil.make_archive(zip_base, 'zip', root_dir='outputs')
print('Created:', zip_file)
print('Download from Kaggle working directory if needed.')
